In [1]:
import os
import fitz  # PyMuPDF for PDF text extraction
import chromadb
from sentence_transformers import SentenceTransformer
import ollama

# Paths
PDF_DIR = r"db\data\ACCOUNTANT"

# Load Sentence Transformer for vector embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB for storing resume vectors
chroma_client = chromadb.PersistentClient(path="./resume_db")
collection = chroma_client.get_or_create_collection(name="resume_vectors")


def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text.strip()


def is_resume(text):
    """Use an LLM to determine if a document is a resume."""
    response = ollama.chat(
        model="gemma2:2b",
        messages=[{"role": "user", "content": f"Is this document a resume? Reply 'Yes' or 'No'.\n{text[:1000]}"}],
    )
    return "yes" in response["message"]["content"].lower()


def index_resumes(pdf_dir):
    """Extract text, filter resumes, generate embeddings, and store in ChromaDB."""
    for root, _, files in os.walk(pdf_dir):
        for file in files:
            if file.endswith(".pdf"):
                file_path = os.path.join(root, file)
                text = extract_text_from_pdf(file_path)
                if not text:
                    continue  # Skip empty PDFs
                
                # Check if it's a resume
                if not is_resume(text):
                    print(f"Skipping (Not a resume): {file}")
                    continue

                # Generate vector embeddings for resumes
                embedding = embedding_model.encode(text).tolist()
                collection.add(
                    ids=[file],
                    embeddings=[embedding],
                    metadatas=[{"file_name": file, "file_path": file_path, "text": text}],
                )
                print(f"Indexed Resume: {file}")

index_resumes(PDF_DIR)

d:\ENTERTERMENT\coding\testing\Power_Grid-PDF-\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed Resume: 10554236.pdf
Indexed Resume: 10674770.pdf
Indexed Resume: 11163645.pdf
Indexed Resume: 11759079.pdf
Indexed Resume: 12065211.pdf
Indexed Resume: 12202337.pdf
Skipping (Not a resume): 12338274.pdf
Indexed Resume: 12442909.pdf
Indexed Resume: 12780508.pdf
Indexed Resume: 12802330.pdf
Indexed Resume: 13072019.pdf
Indexed Resume: 13130984.pdf
Indexed Resume: 13294301.pdf
Indexed Resume: 13491889.pdf
Indexed Resume: 13701259.pdf
Skipping (Not a resume): 14055988.pdf
Indexed Resume: 14126433.pdf
Indexed Resume: 14224370.pdf
Skipping (Not a resume): 14449423.pdf
Indexed Resume: 14470533.pdf
Indexed Resume: 14491649.pdf
Indexed Resume: 14496667.pdf
Indexed Resume: 15289348.pdf
Indexed Resume: 15363277.pdf
Indexed Resume: 15592167.pdf
Indexed Resume: 15821633.pdf
Indexed Resume: 15906625.pdf
Indexed Resume: 16237710.pdf
Indexed Resume: 17306905.pdf
Indexed Resume: 17407184.pdf
Indexed Resume: 17556527.pdf
Skipping (Not a resume): 18132924.pdf
Indexed Resume: 18365791.pdf
Indexed

In [6]:
def search_resumes(query):
    """Search for resumes that match a given query (e.g., experience, gender)."""
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=5)

    if results["ids"]:
        print("\nShortlisted Resumes:")
        for i in range(len(results["ids"][0])):
            file_name = results["metadatas"][0][i]["file_name"]
            file_path = results["metadatas"][0][i]["file_path"]
            text = results["metadatas"][0][i]["text"]

            # Validate results using LLM
            llm_response = ollama.chat(
                model="gemma2:2b",
                messages=[{"role": "user", "content": f"Does this resume match '{query}'? Reply 'Yes' or 'No'.\n{text[:1000]}"}],
            )

            if "yes" in llm_response["message"]["content"].lower():
                print(f"- {file_name} (Path: {file_path})")
    else:
        print("No resumes match the criteria.")
# user_query = "Give me all file name"

# search_resumes(user_query)

user_query = "tell me about the files"
search_resumes(user_query)


Shortlisted Resumes:
- 21338490.pdf (Path: db\data\ACCOUNTANT\21338490.pdf)
